# 📈 Correlación y regresión lineal simple

**Métodos Cuantitativos II · Antropología UAH · 11 de diciembre de 2026**

> 💡 **Pregunta guía:** En los distritos suizos de fines del siglo XIX, ¿cómo se relacionaban la escolaridad y la fecundidad? ¿Qué podemos aprender de esa relación y qué no podemos concluir?

Al terminar podrás describir una asociación con un gráfico, comparar tres coeficientes de correlación, interpretar una regresión lineal y distinguir una predicción dentro del rango de los datos de una extrapolación.

🧭 **Cómo trabajar:** ejecuta los bloques en orden. Antes de cada bloque de análisis, escribe qué esperas observar; después, explica el resultado con palabras y unidades. En Colab, usa un entorno de ejecución **R**.

## 📂 0. Subir el archivo de datos

1. Ten a mano el archivo **`swiss.csv`** entregado para esta clase.
2. En Colab, pulsa **Conectar** y abre **Archivos** (ícono de carpeta a la izquierda).
3. Pulsa **Subir al almacenamiento de la sesión** y selecciona `swiss.csv`. Comprueba que aparezca en el panel con ese nombre.
4. Ejecuta la primera celda de R. Deberías ver seis filas de la base. Si inicias una nueva sesión y el archivo ya no aparece, vuelve a subirlo.

## 🔎 1. Conocer los datos

Cada fila representa uno de **47 distritos francófonos de Suiza** hacia **1888**. `Canton` conserva el nombre de la columna del CSV original; aquí identifica al distrito. Los datos son agregados: un patrón entre distritos no describe necesariamente lo que ocurre entre personas.

| Variable | Qué mide |
|:---|:---|
| `Fertility` | Índice estandarizado de fecundidad; no es un número de hijos ni un porcentaje. |
| `Education` | Porcentaje de reclutas con educación posterior a la primaria. |
| `Agriculture` | Porcentaje de hombres ocupados en agricultura. |
| `Catholic` | Porcentaje de población católica. |
| `Examination` | Porcentaje de reclutas con la mejor calificación en el examen militar. |
| `InfantMortality` | Porcentaje de nacidos vivos que fallecieron antes de cumplir un año. |

In [ ]:
# Importamos el CSV que subiste al almacenamiento de la sesión.
archivo <- if (dir.exists("/content")) "/content/swiss.csv" else "swiss.csv"
if (!file.exists(archivo)) {
  stop("No encuentro swiss.csv. Súbelo desde Archivos en Colab y vuelve a ejecutar esta celda.", call. = FALSE)
}
suiza <- read.csv(archivo, stringsAsFactors = FALSE)
columnas <- c("Canton", "Fertility", "Agriculture", "Examination", "Education", "Catholic", "InfantMortality")
if (!all(columnas %in% names(suiza))) {
  stop("El archivo no tiene las columnas esperadas. Revisa que sea el swiss.csv de la clase.", call. = FALSE)
}

# Si ggplot2 ya está disponible, este bloque no instala nada.
if (!requireNamespace("ggplot2", quietly = TRUE)) {
  install.packages("ggplot2", repos = "https://cloud.r-project.org")
}
library(ggplot2)
theme_set(theme_minimal(base_size = 13))

head(suiza)

In [ ]:
dim(suiza)
summary(suiza[c("Education", "Fertility", "Agriculture")])
colSums(is.na(suiza))

⏸️ **Pausa:** localiza el mínimo y máximo de `Education`. Guárdalos: volveremos a ellos al predecir.

## 👀 2. Mirar antes de calcular

Cada punto es un distrito. Ponemos la escolaridad en el eje horizontal y la fecundidad en el vertical. Lee la nube de izquierda a derecha: ¿sube o baja? ¿Se parece a una recta? ¿Hay puntos apartados?

In [ ]:
nube <- ggplot(suiza, aes(x = Education, y = Fertility)) +
  labs(x = "Reclutas con educación posterior a primaria (%)",
       y = "Índice estandarizado de fecundidad")
nube + geom_point(color = "#176B73", size = 2.5) +
  labs(title = "Escolaridad y fecundidad por distrito")

In [ ]:
destacados <- subset(suiza, Education >= 30)
nube + geom_point(aes(size = Catholic), color = "#176B73", alpha = 0.7) +
  geom_text(data = destacados, aes(label = Canton),
            nudge_y = 3, check_overlap = TRUE, size = 3) +
  scale_size_area(max_size = 7) +
  labs(title = "Una tercera variable: composición religiosa",
       size = "Población católica (%)")

Cambiar el tamaño de los puntos ayuda a formular preguntas sobre una tercera variable. No equivale a controlarla estadísticamente. ¿Qué procesos históricos podrían relacionar educación y fecundidad? ¿Qué datos necesitarías para evaluarlos?

## 📊 3. Medir la asociación

- **Pearson (`r`)** resume qué tan alineados están los puntos con una recta. Puede ser sensible a observaciones extremas.
- **Spearman (`rho`)** aplica la correlación a los rangos. Puede detectar una tendencia que siempre sube o baja aunque no sea recta.
- **Kendall (`tau`)** compara pares: si dos distritos se ordenan en el mismo sentido en ambas variables, el par es concordante; si se ordenan al revés, es discordante.

Los tres van de −1 a 1. El signo indica la dirección. Un valor cercano a cero indica poca asociación del tipo que mide ese coeficiente. Sus magnitudes no son directamente intercambiables.

In [ ]:
metodos <- c("pearson", "spearman", "kendall")
coeficientes <- sapply(metodos, function(metodo) {
  cor(suiza$Education, suiza$Fertility, method = metodo)
})
round(coeficientes, 3)

💬 **Lectura guiada:** los tres signos son negativos. Pearson da aproximadamente −0,664; los coeficientes por rangos también muestran una tendencia descendente. El gráfico ayuda a entender por qué el orden y la alineación lineal no cuentan exactamente la misma historia.

### 🧪 ¿Qué aporta una prueba?

Para Pearson, la hipótesis nula plantea que la correlación lineal poblacional es cero bajo el modelo de la prueba. Lee el coeficiente, su intervalo de confianza y el valor p por separado. En Spearman y Kendall usamos `exact = FALSE` para calcular valores p aproximados con empates.

In [ ]:
cor.test(~ Education + Fertility, data = suiza, method = "pearson")
cor.test(~ Education + Fertility, data = suiza, method = "spearman", exact = FALSE)
cor.test(~ Education + Fertility, data = suiza, method = "kendall", exact = FALSE)

🗣️ **Discute:** un valor p pequeño no mide la fuerza de la relación ni identifica una causa. La interpretación inferencial requiere supuestos como independencia entre observaciones; distritos vecinos podrían estar relacionados. Estos resultados son un ejercicio con datos históricos, no una generalización a cualquier lugar o época.

## 📐 4. Ajustar una regresión lineal simple

La correlación es simétrica. Para predecir con una regresión, en cambio, elegimos un resultado ($Y$) y un predictor ($X$). En `lm(Y ~ X)` la variable a la izquierda es la que queremos explicar o predecir. Mínimos cuadrados elige la recta que minimiza la suma de los residuos al cuadrado. Cada residuo es el valor observado menos el ajustado.

In [ ]:
modelo_educacion <- lm(Fertility ~ Education, data = suiza)
summary(modelo_educacion)
confint(modelo_educacion)

La recta estimada es $\widehat{Y}=79.61-0.862X$.

- **Pendiente:** un punto porcentual adicional de educación se asocia con **0,86 puntos menos** en el índice de fecundidad ajustado. No es una reducción del 0,86 % ni un efecto causal.
- **Intercepto:** 79,61 es el valor que la recta predice para `Education = 0`, apenas fuera del rango observado. Evita darle una interpretación sustantiva automática.
- **$R^2$:** el modelo da aproximadamente 0,44. La recta representa un 44 % de la variación observada de fecundidad en estos datos.

In [ ]:
nube + geom_point(color = "#176B73", size = 2.5) +
  geom_smooth(method = "lm", formula = y ~ x, color = "#A14225",
              fill = "#F1C4AE", se = TRUE) +
  labs(title = "Recta estimada con escolaridad",
       subtitle = "La banda estima la incertidumbre de la media ajustada")

In [ ]:
plot(modelo_educacion, which = 1)

En el gráfico de residuos, busca curvatura o una dispersión que se ensanche. Son pistas para revisar el ajuste, no una prueba de que todos los supuestos se cumplen.

### 🔄 Cambiar el predictor

In [ ]:
modelo_agricultura <- lm(Fertility ~ Agriculture, data = suiza)
summary(modelo_agricultura)
confint(modelo_agricultura)

⏸️ **Pausa:** compara el signo de las pendientes y los $R^2$. ¿Cuál de las dos rectas representa más variación en estos datos? Recuerda que son dos modelos separados: ninguno ajusta por el predictor del otro.

## 🎯 5. Predecir sin perder de vista el rango

Para comparar predicciones, probamos con 10, 50 y 90 puntos porcentuales. El rango observado es de **1 a 53**: 10 y 50 están dentro; 90 queda fuera. R devuelve un número para 90, pero los datos no respaldan directamente esa extrapolación.

In [ ]:
rango <- range(suiza$Education)
nuevos <- data.frame(Education = c(10, 50, 90))
nuevos$fuera_de_rango <- with(nuevos, Education < rango[1] | Education > rango[2])
nuevos

# Media esperada para distritos con ese nivel de educación.
round(predict(modelo_educacion, nuevos, interval = "confidence"), 2)

# Valor de un distrito nuevo con ese nivel de educación.
round(predict(modelo_educacion, nuevos, interval = "prediction"), 2)

El intervalo de predicción es más ancho porque incorpora también la variación de un distrito nuevo alrededor de la media. Ninguno de los intervalos garantiza que la recta siga siendo razonable fuera del rango observado.

## 🤝 6. Actividad en duplas

Una pregunta nueva: **¿cómo se relaciona `InfantMortality` con `Fertility` en estos distritos?**


### 📈 1. Dibuja la nube

Pon la mortalidad infantil en el eje X y la fecundidad en el eje Y. Describe si la nube sube o baja, qué forma tiene y si hay algún punto que te llame la atención.

In [ ]:
# Completa los espacios y quita el # para ejecutar.
# ggplot(suiza, aes(x = ___, y = ___)) + geom_point()
# Lo que veo en el gráfico:

### 🔢 2. Calcula Pearson y Spearman

Calcula ambos coeficientes. ¿Coinciden en la dirección? ¿Cuál destacarías en un informe y qué parte del gráfico respalda tu elección?

In [ ]:
# cor(suiza$InfantMortality, suiza$Fertility, method = "___")
# cor(suiza$InfantMortality, suiza$Fertility, method = "___")
# Elegiría… porque…

### 📐 3. Ajusta e interpreta la recta

Usa la fecundidad como resultado. Reporta el R² y explica cuánto cambia la fecundidad estimada cuando la mortalidad infantil sube un punto porcentual.

In [ ]:
# modelo_infantil <- lm(___ ~ ___, data = suiza)
# summary(modelo_infantil)
# Por cada punto porcentual adicional…
# R²:

### 🎯 4. Haz una predicción

Revisa el rango observado y elige un valor dentro de ese rango. Predice la fecundidad y decide si quieres estimar la media o predecir un distrito nuevo.

In [ ]:
range(suiza$InfantMortality)
# nuevo_distrito <- data.frame(InfantMortality = ___)
# predict(modelo_infantil, newdata = nuevo_distrito, interval = "___")
# Pista: usa "confidence" para la media o "prediction" para un distrito nuevo.
# Mi predicción y lo que representa:

### ✍️ 5. Redacta tu conclusión

Escribe cuatro oraciones. Puedes escribirlas como comentarios de R: deja el `#` al inicio de cada línea.

In [ ]:
# 1. Nuestra pregunta fue…
# 2. En el gráfico observamos…
# 3. Según el modelo estimamos que…
# 4. Un límite de esta interpretación es…

🎟️ **Para salir:** explica en una oración la diferencia entre correlación, pendiente y predicción. Nombra una variable que investigarías antes de plantear una explicación causal.

## 📚 Fuentes

- [Documentación de R: Swiss Fertility and Socioeconomic Indicators (1888)](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/swiss.html).